In [1]:
import rasterio
import matplotlib.pyplot as plt
from s3fs import S3FileSystem
from IPython.display import display, Markdown
import json

from getpass import getpass
secret = "inpe_secret_2024"
# secret = getpass("Secret key: ")

minio = Minio(
    "minio:9000",
    access_key="inpe_admin",
    secret_key=secret,
    secure=False
)

In [3]:
experiment_id = "3939ef6630ce4b748af9f60a1b033480"
short_id = experiment_id[:8]

bucket = "dissmodel-outputs"

base_path = f"experiments/{experiment_id}"

record_path = f"{base_path}/{short_id}.record.json"
profiling_path = f"{base_path}/profiling_{short_id}.md"

# ── 3. CONSTANTES DE USO E SOLO ────────────────────────────────────────────

MANGUE                     = 1
VEGETACAO_TERRESTRE        = 2
MAR                        = 3
AREA_ANTROPIZADA           = 4
SOLO_DESCOBERTO            = 5
SOLO_INUNDADO              = 6
AREA_ANTROPIZADA_INUNDADA  = 7
MANGUE_MIGRADO             = 8
MANGUE_INUNDADO            = 9
VEG_TERRESTRE_INUNDADA     = 10

USO_LABELS = {
    MANGUE: "Mangue",
    VEGETACAO_TERRESTRE: "Veg. Terrestre",
    MAR: "Mar",
    AREA_ANTROPIZADA: "Área Antropizada",
    SOLO_DESCOBERTO: "Solo Descoberto",
    SOLO_INUNDADO: "Solo Inundado",
    AREA_ANTROPIZADA_INUNDADA: "Área Antrop. Inundada",
    MANGUE_MIGRADO: "Mangue Migrado",
    MANGUE_INUNDADO: "Mangue Inundado",
    VEG_TERRESTRE_INUNDADA: "Veg. Terrestre Inundada"
}

USO_COLORS = {
    MANGUE: "#006400",
    VEGETACAO_TERRESTRE: "#808000",
    MAR: "#00008b",
    AREA_ANTROPIZADA: "#ffd700",
    SOLO_DESCOBERTO: "#ffdead",
    SOLO_INUNDADO: "#000000",
    AREA_ANTROPIZADA_INUNDADA: "#323232",
    MANGUE_MIGRADO: "#00ff00",
    MANGUE_INUNDADO: "#ff0000",
    VEG_TERRESTRE_INUNDADA: "#8b4513"
}

# ── 4. LEITURA DO RECORD.JSON ──────────────────────────────────────────────

obj = minio.get_object(bucket, record_path)

record_data = json.loads(obj.read().decode("utf-8"))

print("Record carregado:")
print(json.dumps(record_data, indent=2))

# ── 5. LEITURA DO PROFILING ────────────────────────────────────────────────

obj = minio.get_object(bucket, profiling_path)

profiling_md = obj.read().decode("utf-8")

display(Markdown(profiling_md))

# ── 6. LEITURA DO RASTER ───────────────────────────────────────────────────

# Exemplo:
# s3://dissmodel-outputs/experiments/.../resultado.tif

tif_uri = record_data["output_path"]

# Remove prefixo s3://bucket/
tif_object = tif_uri.replace(f"s3://{bucket}/", "")

obj = minio.get_object(bucket, tif_object)

# Rasterio precisa de arquivo-like
raster_bytes = io.BytesIO(obj.read())

with rasterio.open(raster_bytes) as src:

    data = src.read(1)

    present_values = sorted([
        v for v in np.unique(data)
        if v in USO_COLORS
    ])

    colors_list = [USO_COLORS[v] for v in present_values]
    labels_list = [USO_LABELS[v] for v in present_values]

    cmap = mcolors.ListedColormap(colors_list)

    norm = mcolors.BoundaryNorm(
        np.append(present_values, present_values[-1] + 1),
        cmap.N
    )

    fig, ax = plt.subplots(figsize=(12, 8))

    im = ax.imshow(
        data,
        cmap=cmap,
        norm=norm,
        interpolation='none'
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        ticks=np.array(present_values) + 0.5
    )

    cbar.ax.set_yticklabels(labels_list)
    cbar.set_label('Uso e Cobertura da Terra')

    plt.title(f"Resultado DisSModel - ID: {short_id}")

    plt.axis('off')

    plt.show()

PermissionError: Forbidden